<a href="https://colab.research.google.com/github/priyanshi17112007/AI-NOTES-TOKEN-OPTIMIZER-ENGINE/blob/main/AI_Notes_Token_Optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-genai python-dotenv

In [8]:
import os
import sys
import json
import time
import hashlib
from functools import lru_cache, wraps
from collections import Counter
from typing import Generator, List, Dict, Tuple
from dataclasses import dataclass, field
from google import genai
from google.genai import types
from dotenv import load_dotenv
from google.colab import userdata

load_dotenv()

# --- 100% PURE REAL PRODUCTION MODE ---
SIMULATION_MODE = False

# Core Credential Loading
GEMINI_KEY = userdata.get('GEMINI_API_KEY') if 'google.colab' in sys.modules else os.getenv('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_KEY)

MODEL_NAME = 'gemini-2.5-flash'
CHUNK_SIZE = 500
CACHE_FILE = 'token_cache.json'

# --- FIX #7: Config-Based Pricing Dynamic Load with Safe Fallbacks ---
# Rates per 1,000,000 tokens based on Gemini 2.5 Flash Pay-As-You-Go card pricing
USD_TO_INR = float(os.getenv('USD_TO_INR', 83.50))
RATE_IN_INR = (float(os.getenv('INPUT_RATE_USD', 0.30)) / 1_000_000) * USD_TO_INR
RATE_OUT_INR = (float(os.getenv('OUTPUT_RATE_USD', 2.50)) / 1_000_000) * USD_TO_INR
DAILY_LIMIT_INR = float(os.getenv('DAILY_LIMIT_INR', 50.00))

@dataclass
class ChunkResult:
    chunk_id: int
    text: str
    tokens: int = 0
    summary: str = ""
    cost_inr: float = 0.0
    cached: bool = False
    # --- FIX #9: Structured Metadata Audit Tracing ---
    metadata: Dict = field(default_factory=dict)

@dataclass
class SessionStats:
    # --- FIX #8: Enhanced Request Execution Metrics ---
    api_calls: int = 0
    failed_calls: int = 0
    retry_count: int = 0
    cache_hits: int = 0
    # --- FIX #6: Explicit Staging for Multi-Token Accounting ---
    total_in: int = 0
    total_out: int = 0
    costs_inr: List[float] = field(default_factory=list)

    @property
    def total_cost_inr(self) -> float:
        return sum(self.costs_inr)

    @property
    def cache_rate(self) -> float:
        total = self.api_calls + self.cache_hits
        if not total: return 0.0
        return (self.cache_hits / total) * 100

    def __repr__(self) -> str:
        return (f"API Calls: {self.api_calls} | Retries: {self.retry_count} | "
                f"Input Tokens: {self.total_in} | Output Tokens: {self.total_out} | "
                f"Total Real Cost: ₹{self.total_cost_inr:.5f} | Cache Rate: {self.cache_rate:.1f}%")

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        return result
    return wrapper

def log_call(func):
    @wraps(func)
    def wrapper(self, chunk, *args, **kwargs):
        timestamp = time.strftime('%Y-%m-%d %H:%M:%S')
        generated_tuples = []

        generator = func(self, chunk, *args, **kwargs)
        if generator is not None:
            for update in generator:
                generated_tuples.append(update)
                yield update

        full_output = "".join([str(item[0]) for item in generated_tuples if isinstance(item, tuple)])
        try:
            with open("log.txt", "a", encoding="utf-8") as f:
                f.write(f"[{timestamp}] - Chunk {chunk.chunk_id} Input Words: {len(chunk.text.split())}\n")
                f.write(f"Output: {full_output}\n")
                f.write("-" * 50 + "\n")
        except Exception as e:
            pass
    return wrapper

class TokenOptimizer:
    def __init__(self, daily_limit_inr: float = DAILY_LIMIT_INR):
        self._stats = SessionStats()
        self.limit_inr = daily_limit_inr
        self._cache = self.load_cache()
        self._costs_by_type_inr = { 'summary': 0.0, 'ab_testing': 0.0 }

    @property
    def total_cost_inr(self) -> float:
        return self._stats.total_cost_inr

    @property
    def cache_hit_rate(self) -> str:
        return f"{self._stats.cache_rate:.1f}%"

    # --- FIX #4: Clean sliding windows without Tail Overlap Clones ---
    def chunk_text(self, text: str, overlap: int = 50) -> Generator[str, None, None]:
        words = text.split()
        i = 0
        while i < len(words):
            chunk = words[i:i + CHUNK_SIZE]
            if not chunk:
                break
            yield ' '.join(chunk)
            if i + CHUNK_SIZE >= len(words):
                break
            i += (CHUNK_SIZE - overlap)

    def count_tokens_via_api(self, text: str) -> int:
        try:
            response = client.models.count_tokens(model=MODEL_NAME, contents=text)
            return response.total_tokens
        except Exception:
            return int(len(text.split()) * 1.33) + 10

    def load_cache(self) -> dict:
        if not os.path.exists(CACHE_FILE): return {}
        try:
            with open(CACHE_FILE, 'r') as f: return json.load(f)
        except Exception: return {}

    def _save_cache(self) -> None:
        with open(CACHE_FILE, 'w') as f: json.dump(self._cache, f, indent=2)

    # --- FIX #3: Context Analysis Checks Cache State to Skip Tokens API Hits ---
    def _analyze_chunks(self, text: str, system_instruction: str = None) -> List[ChunkResult]:
        raw_chunks = list(self.chunk_text(text))
        results = []
        for chunk_id, chunk_text in enumerate(raw_chunks):
            # Generate target hash key up front to see if we can save computation
            cache_seed = f"{chunk_text}||{system_instruction or ''}"
            key = hashlib.sha256(cache_seed.encode()).hexdigest()

            if key in self._cache and isinstance(self._cache[key], dict):
                # Pull stored parameters right out of cache architecture
                cached_data = self._cache[key]
                results.append(ChunkResult(
                    chunk_id=chunk_id,
                    text=chunk_text,
                    tokens=cached_data.get("tokens", 0),
                    summary=cached_data.get("summary", ""),
                    cached=True
                ))
            else:
                n_tok = self.count_tokens_via_api(chunk_text)
                results.append(ChunkResult(chunk_id=chunk_id, text=chunk_text, tokens=n_tok, cached=False))
        return results

    def build_report(self, results: List[ChunkResult]) -> Dict:
        chunk_ids = [r.chunk_id for r in results]
        summaries = [r.summary for r in results]
        costs = [r.cost_inr for r in results]
        cached_flags = [r.cached for r in results]

        table_2d = list(zip(chunk_ids, summaries, costs, cached_flags))
        all_words = ' '.join(summaries).lower().split()
        stop_words = {'the', 'a', 'an', 'is', 'in', 'to', 'of', 'and', 'it', 'for', 'this', 'they', '*', '•'}
        word_freq = Counter(w for w in all_words if w not in stop_words and len(w) > 2)

        return {
            'chunks': len(results),
            'total_tokens': sum(r.tokens for r in results),
            'total_cost_inr': sum(costs),
            'cache_hits': sum(1 for c in cached_flags if c),
            'table': table_2d,
            'top_keywords': word_freq.most_common(10)
        }

    @log_call
    @timer
    def summarize_streaming(self, chunk: ChunkResult, system_instruction: str = None, chunk_total: int = 1) -> Generator[Tuple[str, float], None, None]:
        # --- FIX #5: Upgrade Cryptographic Hashing Engine to SHA-256 ---
        cache_seed = f"{chunk.text}||{system_instruction or ''}"
        key = hashlib.sha256(cache_seed.encode()).hexdigest()
        percent = ((chunk.chunk_id + 1) / chunk_total) * 100

        if key in self._cache and isinstance(self._cache[key], dict):
            chunk.cached = True
            self._stats.cache_hits += 1
            chunk.cost_inr = 0.0
            cached_payload = self._cache[key]
            chunk.summary = cached_payload.get("summary", "")
            chunk.tokens = cached_payload.get("tokens", 0)
            for segment in chunk.summary.split(' '):
                yield segment + ' ', percent
            return

        # --- FIX #2: Dynamic Budget Interceptor Protection ---
        # Safeguards system BEFORE making the expensive network request
        estimated_input_cost = chunk.tokens * RATE_IN_INR
        if (self.total_cost_inr + estimated_input_cost) > self.limit_inr:
            raise RuntimeError(f"🛑 Budget Guard Active: Next token window will breach limit threshold of ₹{self.limit_inr:.2f}")

        prompt = f"Summarize in 3 clear bullet points:\n{chunk.text}"
        config_args = {}
        if system_instruction:
            config_args['system_instruction'] = system_instruction

        max_retries = 3
        backoff = 2
        response = None

        for attempt in range(max_retries):
            try:
                self._stats.api_calls += 1
                response = client.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt,
                    config=types.GenerateContentConfig(**config_args) if config_args else None
                )
                break
            except Exception as e:
                self._stats.failed_calls += 1
                if attempt == max_retries - 1:
                    raise e
                if "503" in str(e) or "429" in str(e):
                    self._stats.retry_count += 1
                    print(f"\n⚠️ Rate limit or server busy. Backing off for {backoff}s...")
                    time.sleep(backoff)
                    backoff *= 2
                else:
                    raise e

        result_text = response.text or ""
        metadata = response.usage_metadata

        # --- FIX #1 & #10: Eradicate Inventions, Pull Actual Verified Usage Metrics Only ---
        real_in_tokens = getattr(metadata, 'prompt_token_count', 0) or getattr(metadata, 'input_token_count', 0)
        real_out_tokens = getattr(metadata, 'candidates_token_count', 0) or getattr(metadata, 'output_token_count', 0)

        # Exact ledger modifications
        chunk.cost_inr = (real_in_tokens * RATE_IN_INR) + (real_out_tokens * RATE_OUT_INR)
        chunk.tokens = real_in_tokens
        chunk.summary = result_text
        chunk.cached = False

        # --- FIX #9: Package Audit Payloads Cleanly ---
        chunk.metadata = {
            "prompt_tokens": real_in_tokens,
            "output_tokens": real_out_tokens,
            "total_tokens": real_in_tokens + real_out_tokens
        }

        # --- FIX #6: Separate Structural Token Registries ---
        self._stats.total_in += real_in_tokens
        self._stats.total_out += real_out_tokens
        self._stats.costs_inr.append(chunk.cost_inr)

        # --- FIX #3: Store Complex Dictionary Records in Local Cache ---
        self._cache[key] = {
            "summary": result_text,
            "tokens": real_in_tokens
        }

        category = 'ab_testing' if system_instruction else 'summary'
        self._costs_by_type_inr[category] += chunk.cost_inr

        for segment in result_text.split(' '):
            yield segment + ' ', percent

    def process_notes(self, text_or_path: str, system_instruction: str = None, stream_output: bool = True) -> Dict:
        text = self.load_notes(text_or_path) if os.path.exists(text_or_path) else text_or_path
        if not text: return {'error': 'No text provided'}

        # Pre-analyze checks the cache automatically
        chunks = self._analyze_chunks(text, system_instruction=system_instruction)
        chunk_total = len(chunks)

        for chunk in chunks:
            if chunk.cached:
                # If cached during analysis, just print the summary instantly without calling API
                if stream_output:
                    print(f"\n⚡ [Cache Hit] Chunk [{chunk.chunk_id + 1}/{chunk_total}]:\n{chunk.summary}")
                    print(f"Progress: [{(chunk.chunk_id + 1)/chunk_total * 100:.0f}% Complete]")
                self._stats.cache_hits += 1
                self._stats.costs_inr.append(0.0)
                continue

            if stream_output:
                print(f"\n🌐 [API Call] Processing Chunk [{chunk.chunk_id + 1}/{chunk_total}]:")
            summary_words = []

            for word, pct in self.summarize_streaming(chunk, system_instruction=system_instruction, chunk_total=chunk_total):
                if stream_output:
                    print(word, end='', flush=True)
                summary_words.append(word)

            if stream_output:
                print(f"\nProgress: [{pct:.0f}% Complete]")

        self._save_cache()
        report = self.build_report(chunks)
        report['session_stats'] = str(self._stats)
        return report

    def load_notes(self, path: str) -> str:
        try:
            with open(path, 'r', encoding='utf-8') as f: return f.read()
        except FileNotFoundError: return ""

if __name__ == "__main__":
    optimizer = TokenOptimizer()

    # Clean environmental run traces for baseline verification
    if os.path.exists(CACHE_FILE): os.remove(CACHE_FILE)
    if os.path.exists("log.txt"): os.remove("log.txt")

    print("==================================================")
    print("AI NOTES TOKEN OPTIMIZER ENGINE (100% PRODUCTION) ")
    print("==================================================")

    user_input = input("\nPaste your raw notes here: ").strip()
    if not user_input:
        print("❌ Input blank nahi ho sakta.")
        sys.exit()

    print("\n========== RUN 1: Core Target Processing (Fresh Cache Setup) ==========")
    report1 = optimizer.process_notes(user_input, stream_output=True)

    print('\n========== FINAL GENUINE METRICS REPORT ==========')
    print(f"Total Session Input Tokens: {optimizer._stats.total_in}")
    print(f"Total Session Output Tokens: {optimizer._stats.total_out}")
    print(f"Total Billable Execution Cost: ₹{report1.get('total_cost_inr'):.5f} INR")
    print(f"Aggregated Performance Metrics: {report1.get('session_stats')}")

    time.sleep(2) # Prevent rapid free-tier RPM spikes

    print("\n========== RUN 2: Testing Cache Accuracy (Pure True ₹0.00) ==========")
    report2 = optimizer.process_notes(user_input, stream_output=True)
    print(f"\nSecond run verified execution tracking: ₹{report2.get('total_cost_inr'):.5f} INR")
    print(f"System Optimized Cache Hit Ratio: {optimizer.cache_hit_rate}")

    time.sleep(2) # Cooldown before testing distinct prompts

    # --- 🚀 RUN 3 IS NOW SAFELY RESTORED HERE ---
    print("\n========== RUN 3: Pure A/B Prompt Testing ==========")
    sample_chunk = user_input[:300] if len(user_input) > 300 else user_input

    # Prepare independent chunk targets
    chunk_a = ChunkResult(0, sample_chunk, 0)
    chunk_b = ChunkResult(1, sample_chunk, 0)

    print("\nExecuting Variant A (Bullet point summary)...")
    res_a = "".join([w[0] for w in optimizer.summarize_streaming(chunk_a, system_instruction='You are a bullet-point summarizer.', chunk_total=1)])
    print(res_a.strip())

    time.sleep(2) # Cooldown to protect free-tier RPM limits

    print("\nExecuting Variant B (Tutor structure summary)...")
    res_b = "".join([w[0] for w in optimizer.summarize_streaming(chunk_b, system_instruction='You are an engineering tutor. Explain structural elements.', chunk_total=1)])
    print(res_b.strip())

    # Save the newly generated A/B states to the persistent database cache file
    optimizer._save_cache()
    print("\n✔ Complete execution tracking done successfully.")

AI NOTES TOKEN OPTIMIZER ENGINE (100% PRODUCTION) 

Paste your raw notes here: AI Notes Token Optimizer Engine is a production-oriented Generative AI application that processes large text inputs efficiently using Google's Gemini 2.5 Flash model. The system automatically chunks lengthy content, performs intelligent summarization, tracks token consumption, estimates API costs in real time, and reduces repeated processing through a SHA-256-based caching mechanism.  The project demonstrates practical GenAI engineering concepts including token optimization, prompt engineering, streaming outputs, cost monitoring, retry handling, cache management, budget enforcement, and A/B prompt experimentation. It is designed for students, developers, and researchers who want to understand how production-grade AI applications manage

========== RUN 1: Core Target Processing (Fresh Cache Setup) ==========

🌐 [API Call] Processing Chunk [1/1]:
Here are 3 clear bullet points summarizing the provided text:

*

In [5]:
!rm -f token_cache.json log.txt